# Hugging Face Translation App

Select a Hugging Face translation model and languages, enter text, and get the translation. Supports both **pair-specific** models (e.g. Helsinki-NLP) and **multilingual** models (e.g. NLLB, M2M-100) with configurable source and target languages.

**Setup:** Run this notebook using the project environment so dependencies (e.g. `torch`) resolve for your platform. From the repo root: `uv sync`, then start Jupyter and select the project kernel.

In [ ]:
import gradio as gr
from transformers import pipeline

In [ ]:

MODELS = [
    ("English → French (Helsinki)", "Helsinki-NLP/opus-mt-en-fr"),
    ("English → German (Helsinki)", "Helsinki-NLP/opus-mt-en-de"),
    ("English → Spanish (Helsinki)", "Helsinki-NLP/opus-mt-en-es"),
    ("German → English (Helsinki)", "Helsinki-NLP/opus-mt-de-en"),
    ("French → English (Helsinki)", "Helsinki-NLP/opus-mt-fr-en"),
    ("NLLB 200 (multilingual)", "facebook/nllb-200-distilled-600M"),
    ("M2M-100 418M (multilingual)", "facebook/m2m100_418M"),
]


NLLB_LANGUAGES = {
    "English": "eng_Latn",
    "French": "fra_Latn",
    "German": "deu_Latn",
    "Spanish": "spa_Latn",
    "Italian": "ita_Latn",
    "Portuguese": "por_Latn",
    "Russian": "rus_Cyrl",
    "Japanese": "jpn_Jpan",
    "Chinese (Simplified)": "zho_Hans",
    "Arabic": "arb_Arab",
    "Dutch": "nld_Latn",
}

MODEL_IDS = [m[1] for m in MODELS]
MODEL_LABELS = [m[0] for m in MODELS]

M2M100_LANGUAGES = {
    "English": "en", "French": "fr", "German": "de", "Spanish": "es", "Italian": "it",
    "Portuguese": "pt", "Russian": "ru", "Japanese": "ja", "Chinese (Simplified)": "zh",
    "Arabic": "ar", "Dutch": "nl",
}

def is_multilingual_model(model_id: str) -> bool:
    return "nllb" in model_id.lower() or "m2m100" in model_id.lower()

In [ ]:
_cached_pipeline = None
_cached_key = None

def translate(text: str, model_id: str, source_lang: str, target_lang: str) -> str:
    """Run translation using cached or newly loaded pipeline."""
    global _cached_pipeline, _cached_key
    if not text or not text.strip():
        return ""

    try:
        if is_multilingual_model(model_id):
            # NLLB uses eng_Latn/fra_Latn; M2M100 uses en/fr
            if "m2m100" in model_id.lower():
                src_code = M2M100_LANGUAGES.get(source_lang, "en")
                tgt_code = M2M100_LANGUAGES.get(target_lang, "fr")
            else:
                src_code = NLLB_LANGUAGES.get(source_lang, "eng_Latn")
                tgt_code = NLLB_LANGUAGES.get(target_lang, "fra_Latn")
            key = (model_id, src_code, tgt_code)
            if _cached_key != key or _cached_pipeline is None:
                _cached_pipeline = pipeline(
                    "translation",
                    model=model_id,
                    src_lang=src_code,
                    tgt_lang=tgt_code,
                )
                _cached_key = key
        else:
            # Helsinki / pair-specific: key is model_id only
            key = (model_id,)
            if _cached_key != key or _cached_pipeline is None:
                _cached_pipeline = pipeline("translation", model=model_id)
                _cached_key = key

        result = _cached_pipeline(text)
        if result and isinstance(result, list) and len(result) > 0:
            out = result[0]
            if isinstance(out, dict) and "translation_text" in out:
                return out["translation_text"]
            if isinstance(out, str):
                return out
        return str(result) if result else ""
    except Exception as e:
        return f"Error: {e}"

In [ ]:
def toggle_lang_visibility(model_id):
    """Show source/target dropdowns only for multilingual (NLLB) models."""
    visible = is_multilingual_model(model_id)
    return gr.update(visible=visible), gr.update(visible=visible)

with gr.Blocks(title="HF Translation") as demo:
    gr.Markdown("### Translate text with a Hugging Face model")
    model_dropdown = gr.Dropdown(
        choices=[(label, mid) for label, mid in MODELS],
        value=MODELS[0][1],
        label="Model",
    )
    with gr.Row() as lang_row:
        source_lang = gr.Dropdown(
            choices=list(NLLB_LANGUAGES.keys()),
            value="English",
            label="Source language",
            visible=False,
        )
        target_lang = gr.Dropdown(
            choices=list(NLLB_LANGUAGES.keys()),
            value="French",
            label="Target language",
            visible=False,
        )
    model_dropdown.change(
        fn=toggle_lang_visibility,
        inputs=[model_dropdown],
        outputs=[source_lang, target_lang],
    )
    input_text = gr.Textbox(label="Text to translate", lines=5, placeholder="Enter text...")
    with gr.Row():
        translate_btn = gr.Button("Translate")
        clear_btn = gr.Button("Clear")
    output_text = gr.Textbox(label="Translation", lines=5, interactive=False)

    translate_btn.click(
        fn=translate,
        inputs=[input_text, model_dropdown, source_lang, target_lang],
        outputs=[output_text],
    )
    clear_btn.click(
        fn=lambda: ("", ""),
        inputs=[],
        outputs=[input_text, output_text],
    )

In [ ]:
demo.launch(inbrowser=True)